# RO-057 Route planning on road graphs: every number in the Note

The searches live in `images/roadkit.py`: A* (Dijkstra when the heuristic is 0), the landmark heuristic (ALT) and
contraction hierarchies, on the small town graph of the Note and on the Chicago street graph in
`data/chicago.graphml.gz` (OpenStreetMap, saved with OSMnx). Random trips use seed 0.

In [1]:
import sys, math, random, statistics as st
sys.path.insert(0, 'images')
import roadkit as rk
r1 = lambda x: round(x, 1)

## 1. The town graph (Figures 3 and 6)

In [2]:
At, Al = rk.town('time'), rk.town('length')
for a, b, kind, length in rk.TOWN_ROADS:
    print(f'{a}-{b}: {length} m at {rk.KMH[kind]} km/h = {r1(length / (rk.KMH[kind] / 3.6))} s')
print('one-way:', rk.ONE_WAY, ' arcs:', sum(len(v) for v in At.values()))
short, fast = rk.astar(Al, 'S', 'T'), rk.astar(At, 'S', 'T')
print('shortest by length', '-'.join(short['path']), short['cost'], 'm,', r1(rk.path_cost(At, short['path'])), 's')
print('fastest by time   ', '-'.join(fast['path']), r1(rk.path_cost(Al, fast['path'])), 'm,', r1(fast['cost']), 's')

S-A: 500 m at 50 km/h = 36.0 s
A-B: 500 m at 30 km/h = 60.0 s
B-C: 500 m at 30 km/h = 60.0 s
C-D: 500 m at 30 km/h = 60.0 s
D-T: 500 m at 50 km/h = 36.0 s
A-N: 400 m at 50 km/h = 28.8 s
N-M: 1500 m at 80 km/h = 67.5 s
M-D: 400 m at 50 km/h = 28.8 s
M-L: 1000 m at 80 km/h = 45.0 s
L-T: 650 m at 50 km/h = 46.8 s
B-E: 400 m at 30 km/h = 48.0 s
E-F: 500 m at 30 km/h = 60.0 s
F-C: 400 m at 30 km/h = 48.0 s
one-way: {('F', 'C')}  arcs: 25
shortest by length S-A-B-C-D-T 2500.0 m, 252.0 s
fastest by time    S-A-N-M-D-T 3300.0 m, 197.1 s


## 2. Dijkstra and A* with the straight line at the top speed (Figure 7)

In [3]:
vmax = rk.town_vmax(); print('top speed', vmax, 'm/s')
hs = rk.straight(rk.TOWN_POS, 'T', vmax)
true = rk.dists(rk.reverse(At), 'T')
for u in 'SABN': print(u, 'straight line', r1(math.dist(rk.TOWN_POS[u], rk.TOWN_POS['T'])), 'm, h', r1(hs(u)), 's, true', r1(true[u]), 's')
print('fastest route at the top speed:', r1(3300 / vmax), 's')
for name, h in (('Dijkstra', None), ('A* straight/top', hs)):
    r = rk.astar(At, 'S', 'T', h, trace=True)
    print(name, len(r['order']), 'taken off:', r['order'], r1(r['cost']))
    for s in r['snaps']:
        print('   off', s['cur'], {k: tuple(r1(x) for x in v) for k, v in s['open'].items()})

top speed 22.22222222222222 m/s
S straight line 2500.0 m, h 112.5 s, true 197.1 s
A straight line 2000.0 m, h 90.0 s, true 161.1 s
B straight line 1500.0 m, h 67.5 s, true 156.0 s
N straight line 2039.6 m, h 91.8 s, true 132.3 s
fastest route at the top speed: 148.5 s
Dijkstra 10 taken off: ['S', 'A', 'N', 'B', 'M', 'E', 'C', 'D', 'L', 'T'] 197.1
   off S {'A': (36.0, 0.0, 36.0)}
   off A {'B': (96.0, 0.0, 96.0), 'N': (64.8, 0.0, 64.8)}
   off N {'B': (96.0, 0.0, 96.0), 'M': (132.3, 0.0, 132.3)}
   off B {'M': (132.3, 0.0, 132.3), 'C': (156.0, 0.0, 156.0), 'E': (144.0, 0.0, 144.0)}
   off M {'C': (156.0, 0.0, 156.0), 'E': (144.0, 0.0, 144.0), 'D': (161.1, 0.0, 161.1), 'L': (177.3, 0.0, 177.3)}
   off E {'C': (156.0, 0.0, 156.0), 'D': (161.1, 0.0, 161.1), 'L': (177.3, 0.0, 177.3), 'F': (204.0, 0.0, 204.0)}
   off C {'D': (161.1, 0.0, 161.1), 'L': (177.3, 0.0, 177.3), 'F': (204.0, 0.0, 204.0)}
   off D {'L': (177.3, 0.0, 177.3), 'F': (204.0, 0.0, 204.0), 'T': (197.1, 0.0, 197.1)}
   off 

## 3. The landmark L (Figures 9 and 10)

In [4]:
tab = rk.landmark_tables(At, ['L']); frm, to = tab['L']
ha = rk.alt_heuristic(tab, 'T')
for u in 'SABMNT': print(u, 'to L', r1(to[u]), ' h_L', r1(ha(u)), ' straight', r1(hs(u)), ' true', r1(true[u]))
print('B: 193.8 - 46.8 =', r1(to['B'] - to['T']))
print('consistency on A->N:', r1(ha('A')), '<=', r1(At['A']['N']), '+', r1(ha('N')), '=', r1(At['A']['N'] + ha('N')))
r = rk.astar(At, 'S', 'T', ha, trace=True)
print('ALT', len(r['order']), r['order'], r1(r['cost']))
for s in r['snaps']: print('   off', s['cur'], {k: tuple(r1(x) for x in v) for k, v in s['open'].items()})

S to L 177.3  h_L 130.5  straight 112.5  true 197.1
A to L 141.3  h_L 94.5  straight 90.0  true 161.1
B to L 193.8  h_L 147.0  straight 67.5  true 156.0
M to L 45.0  h_L 1.8  straight 28.8  true 64.8
N to L 112.5  h_L 65.7  straight 91.8  true 132.3
T to L 46.8  h_L 0.0  straight 0.0  true 0.0
B: 193.8 - 46.8 = 147.0
consistency on A->N: 94.5 <= 28.8 + 65.7 = 94.5
ALT 6 ['S', 'A', 'N', 'M', 'D', 'T'] 197.1
   off S {'A': (36.0, 94.5, 130.5)}
   off A {'B': (96.0, 147.0, 243.0), 'N': (64.8, 65.7, 130.5)}
   off N {'B': (96.0, 147.0, 243.0), 'M': (132.3, 1.8, 134.1)}
   off M {'B': (96.0, 147.0, 243.0), 'D': (161.1, 27.0, 188.1), 'L': (177.3, 46.8, 224.1)}
   off D {'B': (96.0, 147.0, 243.0), 'L': (177.3, 46.8, 224.1), 'C': (221.1, 87.0, 308.1), 'T': (197.1, 0.0, 197.1)}
   off T {'B': (96.0, 147.0, 243.0), 'L': (177.3, 46.8, 224.1), 'C': (221.1, 87.0, 308.1)}


## 4. Contraction hierarchy of the town (Figures 11 and 12)

In [5]:
ch = rk.contract(At, rk.CH_ORDER)
print('ranks', ch['rank'])
for v, added, wit in ch['events']:
    print('contract', v, 'shortcuts', [(a, b, r1(c)) for a, b, c in added], 'witnesses', [(a, b, r1(c), r1(d)) for a, b, c, d in wit])
print('A-N-M-D-C:', r1(28.8 + 67.5 + 28.8 + 60))
for s_, t_ in (('S', 'T'), ('S', 'C')):
    q = rk.ch_query(ch, s_, t_)
    print(s_, t_, r1(q['cost']), 'meet', q['top'], q['steps'], 'path', '-'.join(q['path']))
    print('   taken off by both:', {v: r1(q['df'][v] + q['db'][v]) for v in set(q['fwd']) & set(q['bwd'])})
print('ranks along S-A-N-M-D-T:', [ch['rank'][v] for v in 'SANMDT'])

ranks {'E': 0, 'F': 1, 'B': 2, 'C': 3, 'S': 4, 'T': 5, 'L': 6, 'A': 7, 'D': 8, 'N': 9, 'M': 10}
contract E shortcuts [('B', 'F', 108.0)] witnesses [('F', 'B', 108.0, 108.0)]
contract F shortcuts [] witnesses [('B', 'C', 156.0, 60.0)]
contract B shortcuts [('A', 'C', 120.0), ('C', 'A', 120.0)] witnesses []
contract C shortcuts [] witnesses [('D', 'A', 180.0, 125.1), ('A', 'D', 180.0, 125.1)]
contract S shortcuts [] witnesses []
contract T shortcuts [] witnesses [('D', 'L', 82.8, 73.8), ('L', 'D', 82.8, 73.8)]
contract L shortcuts [] witnesses []
contract A shortcuts [] witnesses []
contract D shortcuts [] witnesses []
contract N shortcuts [] witnesses []
contract M shortcuts [] witnesses []
A-N-M-D-C: 185.1
S T 197.1 meet M [(0, 'S'), (1, 'T'), (0, 'A'), (1, 'D'), (0, 'N'), (1, 'L'), (0, 'M'), (1, 'M')] path S-A-N-M-D-T
   taken off by both: {'M': 197.1}
S C 156.0 meet A [(0, 'S'), (1, 'C'), (0, 'A'), (1, 'D'), (0, 'N'), (1, 'M'), (0, 'M'), (1, 'A'), (1, 'N')] path S-A-B-C
   taken off 

## 5. The Chicago street graph (Figures 1, 4, 8 and 9)

In [6]:
C = rk.chicago()
Ct, Cl, pos = C['adj_time'], C['adj_len'], C['pos']
arcs = [(u, v) for u in Ct for v in Ct[u]]
print('saved:', C['raw_nodes'], 'intersections,', C['raw_edges'], 'streets; kept (every one reaches every other):', len(Ct), len(arcs))
print('one-way arcs', sum(1 for u, v in arcs if u not in Ct[v]), '; streets with a speed-limit tag', C['tagged'], 'of', C['raw_edges'])
X, Y = max(p[0] for p in pos.values()), max(p[1] for p in pos.values())
print('extent', r1(X), 'x', r1(Y), 'm; cells of 0.5 m:', round((X / 0.5) * (Y / 0.5)))
cvmax = max(Cl[u][v] / Ct[u][v] for u, v in arcs)
print('top speed', r1(cvmax * 3.6), 'km/h; median', r1(st.median(C['speed_kmh'][e] for e in arcs)), 'km/h')
print('lengths never below the straight line:', all(Cl[u][v] >= math.dist(pos[u], pos[v]) - 1e-6 for u, v in arcs))
sp = [C['speed_kmh'][e] for e in arcs]
print('share of arcs from 33 to 44 km/h:', round(sum(32.5 <= x <= 44.5 for x in sp) / len(sp), 3), '; above 50 km/h:', sum(x > 50.5 for x in sp))

saved: 1670 intersections, 3946 streets; kept (every one reaches every other): 1628 3857
one-way arcs 1409 ; streets with a speed-limit tag 313 of 3946
extent 4869.7 x 4951.1 m; cells of 0.5 m: 96442404
top speed 88.5 km/h; median 35.3 km/h
lengths never below the straight line: True
share of arcs from 33 to 44 km/h: 0.892 ; above 50 km/h: 129


In [7]:
s, t = rk.nearest(pos, (0.12 * X, 0.1 * Y)), rk.nearest(pos, (0.88 * X, 0.9 * Y))
print('start', s, [r1(c) for c in pos[s]], ' goal', t, [r1(c) for c in pos[t]], ' straight line', r1(math.dist(pos[s], pos[t])), 'm')
marks = rk.farthest_landmarks(Ct, 16, sorted(Ct)[0])
tabs = {k: rk.landmark_tables(Ct, marks[:k]) for k in (1, 4, 8, 16)}
cch = rk.contract(Ct)
print('CH shortcuts', len(cch['mid']))
runs = {'Dijkstra': rk.astar(Ct, s, t), 'straight/top': rk.astar(Ct, s, t, rk.straight(pos, t, cvmax)),
        '16 landmarks': rk.astar(Ct, s, t, rk.alt_heuristic(tabs[16], t))}
for k, r in runs.items(): print(k, len(r['order']), r1(r['cost']), 's')
print('intersections on the fastest route:', len(runs['Dijkstra']['path']))
q = rk.ch_query(cch, s, t); print('CH', len(q['fwd']), '+', len(q['bwd']), r1(q['cost']))
dl = rk.astar(Cl, s, t)
print('fastest route', round(rk.path_cost(Cl, runs['Dijkstra']['path'])), 'm; shortest', round(dl['cost']), 'm taking', r1(rk.path_cost(Ct, dl['path'])), 's')
hs_c, ha_c = rk.straight(pos, t, cvmax), rk.alt_heuristic(tabs[16], t)
print('at the start: true', r1(runs['Dijkstra']['cost']), ' straight/top', r1(hs_c(s)), ' landmarks', r1(ha_c(s)))
tc = rk.dists(rk.reverse(Ct), t); us = [u for u in Ct if u != t]
print('average guess / truth: straight/top', round(st.mean(hs_c(u) / tc[u] for u in us), 2), ' landmarks', round(st.mean(ha_c(u) / tc[u] for u in us), 2))
print('stored landmark times:', 16 * len(Ct) * 2)

start 261127648 [680.3, 584.4]  goal 272215873 [3626.3, 4138.3]  straight line 4616.2 m


CH shortcuts 5509
Dijkstra 1526 492.8 s
straight/top 1229 492.8 s
16 landmarks 108 492.8 s
intersections on the fastest route: 42
CH 67 + 92 492.8
fastest route 7047 m; shortest 6717 m taking 573.1 s
at the start: true 492.8  straight/top 187.7  landmarks 434.5
average guess / truth: straight/top 0.41  landmarks 0.93
stored landmark times: 52096


## 6. Averages over 200 random trips (seed 0)

In [8]:
rng = random.Random(0); nodes = sorted(Ct)
acc = {}
for a, b in (tuple(rng.sample(nodes, 2)) for _ in range(200)):
    rr = {'Dijkstra (time)': rk.astar(Ct, a, b), 'A* straight/top (time)': rk.astar(Ct, a, b, rk.straight(pos, b, cvmax)),
          'Dijkstra (length)': rk.astar(Cl, a, b), 'A* straight line (length)': rk.astar(Cl, a, b, rk.straight(pos, b, 1.0))}
    for k in (1, 4, 8, 16): rr[f'A* {k} landmarks'] = rk.astar(Ct, a, b, rk.alt_heuristic(tabs[k], b))
    for k, v in rr.items(): acc.setdefault(k, []).append(len(v['order']))
    qq = rk.ch_query(cch, a, b)
    assert abs(qq['cost'] - rr['Dijkstra (time)']['cost']) < 1e-6
    acc.setdefault('CH query', []).append(len(qq['fwd']) + len(qq['bwd']))
for k, v in acc.items(): print(f'{k:28s} {st.mean(v):7.1f}')

Dijkstra (time)                861.1
A* straight/top (time)         528.1
Dijkstra (length)              854.5
A* straight line (length)      240.4
A* 1 landmarks                 398.2
A* 4 landmarks                 115.3
A* 8 landmarks                  91.7
A* 16 landmarks                 51.6
CH query                        90.0
